# H12 - Confidence-Threshold Sweep / Calibration

This notebook addresses H12.1, H12.2, and H12.3.

- **H12.1:** Load H10 prediction/probability artifacts and sweep `(tau_low, tau_high)` pairs for the runtime 3-class verdict mapping.
- **H12.2:** Plot a reliability diagram for predicted scam probability vs observed scam frequency. If logits are available, fit a temperature-scaling parameter on the validation split first.
- **H12.3:** Save the selected thresholds, optional temperature parameter, decision markdown, and runtime-consumable JSON constants.

This notebook does **not** run model inference. It only consumes H10 CSV outputs from Google Drive.


## Install

The sweep is lightweight and runs on CPU.


In [ ]:
%pip install -q pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.1 matplotlib==3.9.0 scipy==1.13.1


## Drive Paths

H10 must write its binary prediction/probability CSV to Drive before this notebook can run.

Expected H10 artifact:

```text
/content/drive/MyDrive/GemScan/notebooks/_results/h10_baseline_predictions.csv
```

Expected schema:

```text
id,text,true_label,true_verdict,model_tier,safe_prob,scam_prob,predicted_verdict,source,split
```

`true_verdict` is preferred. `true_label` is accepted as a fallback. Optional logits columns can be included as `safe_logit,scam_logit` for temperature scaling. H12 derives the runtime `suspicious` verdict from the `scam_prob` threshold band; H10 no longer emits `suspicious_prob`.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

NOTEBOOKS_ROOT = Path("/content/drive/MyDrive/GemScan/notebooks")
DATA_DIR = NOTEBOOKS_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SCRUBBED_DIR = DATA_DIR / "scrubbed"
RESULTS_DIR = NOTEBOOKS_ROOT / "_results"
FIXTURES_DIR = DATA_DIR / "fixtures"
PROMPTS_DIR = DATA_DIR / "prompts"

for directory in [PROCESSED_DIR, SCRUBBED_DIR, RESULTS_DIR, FIXTURES_DIR, PROMPTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_H10_ARTIFACT = RESULTS_DIR / "h10_baseline_predictions.csv"
H10_ARTIFACT_PATH = Path(os.environ.get("GEMSCAN_H10_PREDICTIONS", str(EXPECTED_H10_ARTIFACT)))

SWEEP_RESULTS_PATH = RESULTS_DIR / "h12_threshold_sweep.csv"
BEST_METRICS_PATH = RESULTS_DIR / "h12_best_threshold_metrics.json"
RELIABILITY_PLOT_PATH = RESULTS_DIR / "h12_reliability_diagram.png"
DECISION_SUMMARY_PATH = RESULTS_DIR / "h12_threshold_decision.md"
RUNTIME_CONSTANTS_PATH = RESULTS_DIR / "h12_runtime_threshold_constants.json"

LABELS = ["safe", "suspicious", "scam"]
BINARY_LABELS = ["safe", "scam"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}
binary_label2id = {label: idx for idx, label in enumerate(BINARY_LABELS)}

print("H10_ARTIFACT_PATH", H10_ARTIFACT_PATH)
print("RESULTS_DIR", RESULTS_DIR)


## Load H10 Predictions

This cell fails fast if H10 has not produced the required CSV. The failure message is intentionally explicit so H10 can be updated to write the handoff file.


In [ ]:
import numpy as np
import pandas as pd

REQUIRED_BASE_COLUMNS = {"id", "text", "model_tier", "safe_prob", "scam_prob"}
OPTIONAL_CONTEXT_COLUMNS = ["predicted_verdict", "source", "split"]
OPTIONAL_LOGIT_COLUMNS = ["safe_logit", "scam_logit"]

if not H10_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        "Missing H10 prediction/probability artifact. "
        f"Expected {H10_ARTIFACT_PATH}. "
        "H10 must produce a CSV with columns: "
        "id,text,true_label,true_verdict,model_tier,safe_prob,scam_prob,"
        "predicted_verdict,source,split. Optional temperature-scaling logits: "
        "safe_logit,scam_logit."
    )

predictions = pd.read_csv(H10_ARTIFACT_PATH)
missing = REQUIRED_BASE_COLUMNS - set(predictions.columns)
if missing:
    raise ValueError(f"H10 artifact is missing required columns: {sorted(missing)}")
if "true_verdict" not in predictions.columns and "true_label" not in predictions.columns:
    raise ValueError("H10 artifact must contain true_verdict or true_label.")


def normalize_label(value):
    raw = str(value).lower().strip()
    mapping = {
        "ham": "safe",
        "legitimate": "safe",
        "benign": "safe",
        "safe": "safe",
        "0": "safe",
        "spam": "scam",
        "phishing": "scam",
        "fraud": "scam",
        "scam": "scam",
        "malicious": "scam",
        "1": "scam",
    }
    return mapping.get(raw)


label_source = "true_verdict" if "true_verdict" in predictions.columns else "true_label"
predictions["true_verdict_norm"] = predictions[label_source].map(normalize_label)
predictions = predictions.dropna(subset=["true_verdict_norm", "scam_prob", "safe_prob"]).copy()
predictions["true_binary_id"] = predictions["true_verdict_norm"].map(binary_label2id)

prob_cols = ["safe_prob", "scam_prob"]
for column in prob_cols:
    predictions[column] = pd.to_numeric(predictions[column], errors="coerce")
predictions = predictions.dropna(subset=prob_cols).copy()

prob_sum = predictions[prob_cols].sum(axis=1)
bad_prob_rows = predictions[(prob_sum <= 0) | (predictions[prob_cols] < 0).any(axis=1)]
if len(bad_prob_rows):
    raise ValueError(f"Found {len(bad_prob_rows)} rows with invalid probabilities.")
if not np.allclose(prob_sum, 1.0, atol=1e-3):
    predictions[prob_cols] = predictions[prob_cols].div(prob_sum, axis=0)
    print("Renormalized binary probability columns because row sums were not exactly 1.0.")

if "split" not in predictions.columns:
    predictions["split"] = "heldout"
predictions["split"] = predictions["split"].fillna("heldout").astype(str).str.lower().str.strip()

print("loaded rows", len(predictions))
print("columns", predictions.columns.tolist())
print("splits")
print(predictions["split"].value_counts())
print("labels")
print(predictions["true_verdict_norm"].value_counts())
predictions.head()


## Optional Temperature Scaling

If H10 writes logits, this cell fits a single temperature on the validation split and replaces the probability columns used by H12 with calibrated probabilities. If logits are absent, H12 continues with H10 probabilities and records that calibration was skipped.


In [ ]:
from scipy.optimize import minimize_scalar


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def negative_log_likelihood(logits, labels, temperature):
    probs = softmax(logits / temperature)
    clipped = np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0)
    return -np.mean(np.log(clipped))


has_logits = all(column in predictions.columns for column in OPTIONAL_LOGIT_COLUMNS)
temperature = None
temperature_note = "Skipped temperature scaling because H10 did not provide safe_logit,scam_logit."
score_df = predictions.copy()

if has_logits:
    for column in OPTIONAL_LOGIT_COLUMNS:
        score_df[column] = pd.to_numeric(score_df[column], errors="coerce")
    score_df = score_df.dropna(subset=OPTIONAL_LOGIT_COLUMNS).copy()
    validation_df = score_df[score_df["split"].isin(["validation", "val", "dev"])].copy()
    if validation_df.empty:
        temperature_note = "Skipped temperature scaling because logits were present but no validation split was available."
    else:
        val_logits = validation_df[OPTIONAL_LOGIT_COLUMNS].to_numpy(dtype=float)
        val_labels = validation_df["true_binary_id"].to_numpy(dtype=int)
        result = minimize_scalar(
            lambda t: negative_log_likelihood(val_logits, val_labels, t),
            bounds=(0.05, 10.0),
            method="bounded",
        )
        temperature = float(result.x)
        calibrated = softmax(score_df[OPTIONAL_LOGIT_COLUMNS].to_numpy(dtype=float) / temperature)
        score_df[["safe_prob", "scam_prob"]] = calibrated
        temperature_note = f"Fit temperature={temperature:.4f} on {len(validation_df)} validation rows."

print(temperature_note)


## Threshold Sweep

Runtime mapping, per H12.1:

```python
scam if scam_prob >= tau_high
suspicious if tau_low <= scam_prob < tau_high
safe otherwise
```

Primary H12.1 objective:

```text
cost = 1.0 * FN + 0.2 * FP
```

For this binary H9/H10 ground truth:

```text
FN = true scam predicted safe
FP = true safe predicted scam
```

The `suspicious` band is still reported because it affects product usefulness, but it is not part of the required H12.1 objective. The sweep also records diagnostic policy metrics so we can see when a threshold pair wins the spec objective by overusing the middle band.


In [ ]:
from sklearn.metrics import confusion_matrix

# H12.1 derives a runtime 3-class verdict from binary H10 probabilities.
# The primary required objective is exactly: cost = 1.0 * FN + 0.2 * FP.
# With binary H9/H10 labels, FN means true scam -> safe, and FP means true safe -> scam.
H12_FN_WEIGHT = 1.0
H12_FP_WEIGHT = 0.2

# Diagnostic policy metrics. These do not replace the H12.1 objective.
SOFT_FP_SUSPICIOUS_WEIGHT = 0.10        # true safe -> suspicious: user friction
SCAM_ESCALATION_WEIGHT = 0.05           # true scam -> suspicious: caught, but not decisive
MAX_SUSPICIOUS_RATE = 0.25
SUSPICIOUS_OVERAGE_WEIGHT = 0.50        # diagnostic penalty per row over max suspicious rate
MAX_HARD_FALSE_POSITIVE_RATE = 0.05
MIN_SCAM_CAPTURE_RATE = 0.85            # true scam -> suspicious or scam


def threshold_verdicts(scam_prob, tau_low, tau_high):
    scam_prob = np.asarray(scam_prob, dtype=float)
    verdicts = np.full(scam_prob.shape, "safe", dtype=object)
    verdicts[(scam_prob >= tau_low) & (scam_prob < tau_high)] = "suspicious"
    verdicts[scam_prob >= tau_high] = "scam"
    return verdicts


def safe_div(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


def metric_row(df, tau_low, tau_high, model_tier):
    y_true = df["true_verdict_norm"].to_numpy()
    y_pred = threshold_verdicts(df["scam_prob"].to_numpy(), tau_low, tau_high)
    n = int(len(df))

    true_safe = y_true == "safe"
    true_scam = y_true == "scam"
    pred_safe = y_pred == "safe"
    pred_suspicious = y_pred == "suspicious"
    pred_scam = y_pred == "scam"

    scam_missed_safe = int((true_scam & pred_safe).sum())
    scam_escalated_suspicious = int((true_scam & pred_suspicious).sum())
    scam_caught_scam = int((true_scam & pred_scam).sum())
    safe_passed_safe = int((true_safe & pred_safe).sum())
    safe_escalated_suspicious = int((true_safe & pred_suspicious).sum())
    safe_flagged_scam = int((true_safe & pred_scam).sum())

    suspicious_count = int(pred_suspicious.sum())
    suspicious_rate = safe_div(suspicious_count, n)
    suspicious_overage = max(0.0, suspicious_rate - MAX_SUSPICIOUS_RATE)

    false_negative = scam_missed_safe
    hard_false_positive = safe_flagged_scam
    soft_false_positive = safe_escalated_suspicious
    scam_capture_count = scam_escalated_suspicious + scam_caught_scam
    decisive_count = int((pred_safe | pred_scam).sum())

    cost = H12_FN_WEIGHT * scam_missed_safe + H12_FP_WEIGHT * safe_flagged_scam
    policy_cost = (
        H12_FN_WEIGHT * scam_missed_safe
        + H12_FP_WEIGHT * safe_flagged_scam
        + SOFT_FP_SUSPICIOUS_WEIGHT * safe_escalated_suspicious
        + SCAM_ESCALATION_WEIGHT * scam_escalated_suspicious
        + SUSPICIOUS_OVERAGE_WEIGHT * suspicious_overage * n
    )

    policy_ok = (
        safe_div(hard_false_positive, int(true_safe.sum())) <= MAX_HARD_FALSE_POSITIVE_RATE
        and safe_div(scam_capture_count, int(true_scam.sum())) >= MIN_SCAM_CAPTURE_RATE
        and suspicious_rate <= MAX_SUSPICIOUS_RATE
    )

    return {
        "model_tier": model_tier,
        "tau_low": float(tau_low),
        "tau_high": float(tau_high),
        "cost": float(cost),
        "policy_cost": float(policy_cost),
        "policy_ok": bool(policy_ok),
        "false_negative": false_negative,
        "hard_false_positive": hard_false_positive,
        "soft_false_positive": soft_false_positive,
        "scam_missed_safe": scam_missed_safe,
        "scam_escalated_suspicious": scam_escalated_suspicious,
        "scam_caught_scam": scam_caught_scam,
        "safe_passed_safe": safe_passed_safe,
        "safe_escalated_suspicious": safe_escalated_suspicious,
        "safe_flagged_scam": safe_flagged_scam,
        "scam_miss_rate": safe_div(scam_missed_safe, int(true_scam.sum())),
        "scam_capture_rate": safe_div(scam_capture_count, int(true_scam.sum())),
        "hard_false_positive_rate": safe_div(hard_false_positive, int(true_safe.sum())),
        "soft_false_positive_rate": safe_div(soft_false_positive, int(true_safe.sum())),
        "suspicious_rate": suspicious_rate,
        "decisive_rate": safe_div(decisive_count, n),
        "n": n,
    }


def sort_sweep(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.sort_values(
        [
            "cost",
            "false_negative",
            "hard_false_positive",
            "scam_capture_rate",
            "suspicious_rate",
            "decisive_rate",
            "tau_high",
            "tau_low",
        ],
        ascending=[True, True, True, False, True, False, True, True],
    ).reset_index(drop=True)


candidate_splits = ["validation", "val", "dev", "heldout", "test"]
heldout_df = score_df[score_df["split"].isin(candidate_splits)].copy()
if heldout_df.empty:
    heldout_df = score_df[~score_df["split"].isin(["train", "training"])].copy()
if heldout_df.empty:
    heldout_df = score_df.copy()

taus = np.round(np.arange(0.05, 1.00, 0.05), 2)
rows = []
for model_tier, tier_df in heldout_df.groupby("model_tier", sort=True):
    for tau_low in taus:
        for tau_high in taus:
            if tau_low < tau_high:
                rows.append(metric_row(tier_df, tau_low, tau_high, model_tier))

sweep = sort_sweep(pd.DataFrame(rows))
best_by_tier = sort_sweep(
    sweep.groupby("model_tier", group_keys=False).head(1).reset_index(drop=True)
)
best = best_by_tier.iloc[0].to_dict()
best_model_tier = best["model_tier"]
best_tier_df = heldout_df[heldout_df["model_tier"] == best_model_tier].copy()
best_pred = threshold_verdicts(best_tier_df["scam_prob"].to_numpy(), best["tau_low"], best["tau_high"])
best_cm = confusion_matrix(best_tier_df["true_verdict_norm"], best_pred, labels=LABELS)

sweep.to_csv(SWEEP_RESULTS_PATH, index=False)

print("heldout rows", len(heldout_df))
print("heldout rows by tier")
print(heldout_df["model_tier"].value_counts())
print("best by tier")
display(best_by_tier)
print("selected best", best)
print(pd.DataFrame(best_cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS]))
print("saved", SWEEP_RESULTS_PATH)
sweep.head(10)


## Reliability Diagram

This plots H10/H12 scam probability against observed scam frequency. The selected thresholds are overlaid as vertical lines.


In [ ]:
import matplotlib.pyplot as plt


def reliability_bins(df, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    work = df.copy()
    work["bin"] = pd.cut(work["scam_prob"], bins=bins, include_lowest=True, right=True)
    grouped = work.groupby("bin", observed=False)
    out = grouped.agg(
        mean_predicted_scam_prob=("scam_prob", "mean"),
        observed_scam_frequency=("true_verdict_norm", lambda s: float((s == "scam").mean())),
        n=("id", "count"),
    ).reset_index()
    return out.dropna(subset=["mean_predicted_scam_prob", "observed_scam_frequency"])


reliability_frames = []
for model_tier, tier_df in heldout_df.groupby("model_tier", sort=True):
    tier_reliability = reliability_bins(tier_df, n_bins=10)
    tier_reliability["model_tier"] = model_tier
    reliability_frames.append(tier_reliability)
reliability = pd.concat(reliability_frames, ignore_index=True) if reliability_frames else pd.DataFrame()

model_tiers = sorted(heldout_df["model_tier"].dropna().astype(str).unique())
fig, axes = plt.subplots(1, len(model_tiers), figsize=(7 * max(1, len(model_tiers)), 6), squeeze=False)
for ax, model_tier in zip(axes.ravel(), model_tiers):
    tier_reliability = reliability[reliability["model_tier"] == model_tier]
    tier_best = best_by_tier[best_by_tier["model_tier"] == model_tier].iloc[0]
    ax.plot([0, 1], [0, 1], linestyle="--", color="0.6", label="perfect calibration")
    ax.plot(
        tier_reliability["mean_predicted_scam_prob"],
        tier_reliability["observed_scam_frequency"],
        marker="o",
        label=f"{model_tier} scam probability",
    )
    ax.axvline(tier_best["tau_low"], color="#1f77b4", linestyle=":", label=f"tau_low={tier_best['tau_low']:.2f}")
    ax.axvline(tier_best["tau_high"], color="#d62728", linestyle=":", label=f"tau_high={tier_best['tau_high']:.2f}")
    ax.set_xlabel("Predicted scam probability")
    ax.set_ylabel("Observed scam frequency")
    ax.set_title(f"H12 reliability diagram - {model_tier}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
fig.tight_layout()
fig.savefig(RELIABILITY_PLOT_PATH, dpi=160)
plt.show()

print("saved", RELIABILITY_PLOT_PATH)
reliability


## Decision Output

Save artifacts for runtime integration and later review. Results are marked provisional when they come from `notebooks` paths rather than official H10 outputs.


In [ ]:
import json
from datetime import datetime, timezone

provisional = "notebooks" in str(H10_ARTIFACT_PATH)
status = "provisional" if provisional else "official-candidate"
cm_df = pd.DataFrame(best_cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS])
generated_at = datetime.now(timezone.utc).isoformat()

tier_metrics = []
for _, row in best_by_tier.iterrows():
    tier = row["model_tier"]
    tier_df = heldout_df[heldout_df["model_tier"] == tier].copy()
    tier_pred = threshold_verdicts(tier_df["scam_prob"].to_numpy(), row["tau_low"], row["tau_high"])
    tier_cm = confusion_matrix(tier_df["true_verdict_norm"], tier_pred, labels=LABELS)
    tier_metrics.append(
        {
            "model_tier": str(tier),
            "rows_evaluated": int(len(tier_df)),
            "tau_low": float(row["tau_low"]),
            "tau_high": float(row["tau_high"]),
            "cost": float(row["cost"]),
            "policy_cost": float(row["policy_cost"]),
            "policy_ok": bool(row["policy_ok"]),
            "false_negative": int(row["false_negative"]),
            "hard_false_positive": int(row["hard_false_positive"]),
            "soft_false_positive": int(row["soft_false_positive"]),
            "scam_miss_rate": float(row["scam_miss_rate"]),
            "scam_capture_rate": float(row["scam_capture_rate"]),
            "hard_false_positive_rate": float(row["hard_false_positive_rate"]),
            "soft_false_positive_rate": float(row["soft_false_positive_rate"]),
            "suspicious_rate": float(row["suspicious_rate"]),
            "decisive_rate": float(row["decisive_rate"]),
            "confusion_matrix": tier_cm.tolist(),
        }
    )

metrics_payload = {
    "task": "H12",
    "status": status,
    "source_artifact": str(H10_ARTIFACT_PATH),
    "selected_model_tier": str(best_model_tier),
    "rows_evaluated": int(len(heldout_df)),
    "splits_evaluated": sorted(heldout_df["split"].unique().tolist()),
    "tau_low": float(best["tau_low"]),
    "tau_high": float(best["tau_high"]),
    "temperature": temperature,
    "temperature_note": temperature_note,
    "cost": float(best["cost"]),
    "policy_cost": float(best["policy_cost"]),
    "policy_ok": bool(best["policy_ok"]),
    "false_negative": int(best["false_negative"]),
    "hard_false_positive": int(best["hard_false_positive"]),
    "soft_false_positive": int(best["soft_false_positive"]),
    "scam_miss_rate": float(best["scam_miss_rate"]),
    "scam_capture_rate": float(best["scam_capture_rate"]),
    "hard_false_positive_rate": float(best["hard_false_positive_rate"]),
    "soft_false_positive_rate": float(best["soft_false_positive_rate"]),
    "suspicious_rate": float(best["suspicious_rate"]),
    "decisive_rate": float(best["decisive_rate"]),
    "confusion_matrix_labels": LABELS,
    "confusion_matrix": best_cm.tolist(),
    "best_by_tier": tier_metrics,
    "generated_at": generated_at,
}

runtime_constants = {
    "gemscan_thresholds_version": "h12-v0",
    "status": status,
    "selected_model_tier": str(best_model_tier),
    "tau_low": float(best["tau_low"]),
    "tau_high": float(best["tau_high"]),
    "temperature": temperature,
    "verdict_rule": {
        "safe": "scam_prob < tau_low",
        "suspicious": "tau_low <= scam_prob < tau_high",
        "scam": "scam_prob >= tau_high",
    },
    "source_artifact": str(H10_ARTIFACT_PATH),
    "generated_at": generated_at,
}

BEST_METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2) + "\n")
RUNTIME_CONSTANTS_PATH.write_text(json.dumps(runtime_constants, indent=2) + "\n")

tier_sections = []
for _, row in best_by_tier.iterrows():
    tier_sections.append(
        "\n".join(
            [
                f"### {row['model_tier']}",
                "",
                f"- `tau_low`: `{row['tau_low']:.2f}`",
                f"- `tau_high`: `{row['tau_high']:.2f}`",
                f"- H12.1 cost: `{row['cost']:.4f}`",
                f"- Diagnostic policy cost: `{row['policy_cost']:.4f}`",
                f"- Diagnostic policy constraints passed: `{bool(row['policy_ok'])}`",
                f"- Scam capture rate: `{row['scam_capture_rate']:.4f}`",
                f"- Scam miss rate: `{row['scam_miss_rate']:.4f}`",
                f"- Hard false-positive rate: `{row['hard_false_positive_rate']:.4f}`",
                f"- Soft false-positive rate: `{row['soft_false_positive_rate']:.4f}`",
                f"- Suspicious output rate: `{row['suspicious_rate']:.4f}`",
                f"- Decisive output rate: `{row['decisive_rate']:.4f}`",
            ]
        )
    )
tier_summary = "\n\n".join(tier_sections)

summary = f"""# H12 Threshold Sweep Decision

Status: **{status}**

Source artifact: `{H10_ARTIFACT_PATH}`

Selected model tier: `{best_model_tier}`

Selected thresholds:

- `tau_low`: `{best['tau_low']:.2f}`
- `tau_high`: `{best['tau_high']:.2f}`
- `temperature`: `{temperature if temperature is not None else 'not fit'}`

Temperature scaling: {temperature_note}

H12.1 selection objective:

```text
cost = {H12_FN_WEIGHT} * (true scam -> safe) + {H12_FP_WEIGHT} * (true safe -> scam)
```

Diagnostic policy weights, reported but not used to select the H12.1 winner:

- Safe predicted suspicious: `{SOFT_FP_SUSPICIOUS_WEIGHT}`
- Scam predicted suspicious: `{SCAM_ESCALATION_WEIGHT}`
- Suspicious overage above `{MAX_SUSPICIOUS_RATE:.2f}`: `{SUSPICIOUS_OVERAGE_WEIGHT}` per row-equivalent

Held-out rows evaluated: `{len(heldout_df)}`

## Best Thresholds By Tier

{tier_summary}

## Selected Tier Metrics

- H12.1 cost: `{best['cost']:.4f}`
- Diagnostic policy cost: `{best['policy_cost']:.4f}`
- Diagnostic policy constraints passed: `{bool(best['policy_ok'])}`
- False negatives, true scam -> safe: `{int(best['false_negative'])}`
- Hard false positives, true safe -> scam: `{int(best['hard_false_positive'])}`
- Soft false positives, true safe -> suspicious: `{int(best['soft_false_positive'])}`
- Scam miss rate: `{best['scam_miss_rate']:.4f}`
- Scam capture rate, true scam -> suspicious/scam: `{best['scam_capture_rate']:.4f}`
- Hard false-positive rate: `{best['hard_false_positive_rate']:.4f}`
- Soft false-positive rate: `{best['soft_false_positive_rate']:.4f}`
- Suspicious output rate: `{best['suspicious_rate']:.4f}`
- Decisive output rate: `{best['decisive_rate']:.4f}`

Confusion matrix labels: `{LABELS}`

```text
{cm_df.to_string()}
```

Artifacts:

- Sweep CSV: `{SWEEP_RESULTS_PATH}`
- Reliability diagram: `{RELIABILITY_PLOT_PATH}`
- Metrics JSON: `{BEST_METRICS_PATH}`
- Runtime constants JSON: `{RUNTIME_CONSTANTS_PATH}`

Provisional note: results produced from `notebooks` paths are scaffolding outputs and must be rerun against official H10 predictions before runtime constants are baked into Swift/agent code.
"""

DECISION_SUMMARY_PATH.write_text(summary)

print("saved", BEST_METRICS_PATH)
print("saved", RUNTIME_CONSTANTS_PATH)
print("saved", DECISION_SUMMARY_PATH)
print(summary)
